# v1 Kaggle Spike — Clinical Transcript -> SOAP JSON

**Goal:** prove the SFT pipeline end-to-end on Kaggle T4.

**What this proves**
- Repo + secret + Unsloth install works on Kaggle
- Qwen 2.5 3B Instruct + 4-bit + LoRA trains on toy clinical data
- Adapter saves to `/kaggle/working/adapter_v0`
- Eval loop reports 4 metrics (parse / schema / CC overlap / evidence grounding)

**What this does NOT prove**
- Template-awareness — only one output spec (SOAP) in v1; deferred to v2
- Generalisation — toy transcripts are short and synthetic
- Production quality — n=10 eval, 20 train steps

**Success criteria** (rough): json_parse >= 0.8, schema_keys >= 0.8, evidence_grounding >= 0.5.

In [ ]:
# Cell 2 — env check
import sys
print('Python:', sys.version)
!nvidia-smi

In [ ]:
# Cell 3 — clone repo via GITHUB_PAT secret
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

In [ ]:
%%capture
!pip install -q unsloth
# Replace stable Unsloth with nightly to fix transformers>=4.46 incompat
# (AttributeError: 'int' object has no attribute 'mean' in training_step)
!pip uninstall unsloth -y
!pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q -e .

In [ ]:
# Cell 5 — generate toy data (deterministic)
!python scripts/gen_toy.py --n 100 --seed 0 --out-dir data/
!ls -la data/

In [ ]:
# Cell 6 — load Qwen 2.5 3B Instruct, 4-bit
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print('loaded', MODEL_NAME)

In [ ]:
# Cell 7 — attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=0,
)

In [ ]:
# Cell 8 — format toy data with chat template
import json
from datasets import Dataset
from src.prompts import build_messages

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_rows = load_jsonl('data/toy.train.jsonl')
eval_rows  = load_jsonl('data/toy.eval.jsonl')

def to_text(row):
    msgs = build_messages(row['transcript'])
    msgs.append({'role':'assistant',
                 'content': json.dumps(row['soap'], ensure_ascii=False)})
    return tokenizer.apply_chat_template(msgs, tokenize=False)

train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_rows])
print('n_train =', len(train_ds))
print('---example 0 (first 600 chars)---')
print(train_ds[0]['text'][:600])

In [ ]:
# Cell 9 — SFTTrainer config
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=TrainingArguments(
        output_dir='/kaggle/working/sft_out',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=20,
        learning_rate=2e-4,
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=0,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        report_to='none',
    ),
)

In [ ]:
# Cell 10 — train
trainer.train()

## Eval loop

Four metrics on 10 held-out examples:
1. **json_parse_rate** — output is valid JSON
2. **schema_keys_rate** — top-level has subjective/objective/assessment/plan
3. **cc_overlap_mean** — Jaccard on chief_complaint.text tokens
4. **evidence_grounding_rate** — fraction of leaf evidence_spans that are substrings of the transcript (hallucination guardrail)

In [ ]:
# Cell 12 — eval loop
from src.toy_data import _collect_spans
from src.prompts import build_messages

FastLanguageModel.for_inference(model)

def generate(transcript: str) -> str:
    msgs = build_messages(transcript)
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def score(pred_str: str, gold: dict, transcript: str) -> dict:
    # (1) JSON parse
    try:
        pred = json.loads(pred_str)
    except Exception:
        return {'parse': 0, 'keys': 0, 'cc': 0.0, 'ground': 0.0}
    # (2) schema keys
    keys = int(set(pred.keys()) >= {'subjective','objective','assessment','plan'})
    # (3) CC token overlap (Jaccard)
    try:
        p_cc = (pred['subjective']['chief_complaint']['text'] or '').lower()
        g_cc = (gold['subjective']['chief_complaint']['text'] or '').lower()
        a, b = set(p_cc.split()), set(g_cc.split())
        cc = len(a & b) / max(len(a | b), 1)
    except Exception:
        cc = 0.0
    # (4) evidence grounding
    spans = [s for s in _collect_spans(pred) if s]
    if not spans:
        ground = 0.0
    else:
        ground = sum(1 for s in spans if s in transcript) / len(spans)
    return {'parse': 1, 'keys': keys, 'cc': cc, 'ground': ground}

results = []
preds = []
for row in eval_rows:
    p = generate(row['transcript'])
    preds.append(p)
    results.append(score(p, row['soap'], row['transcript']))

n = len(results)
summary = {
    'n': n,
    'json_parse_rate':         sum(r['parse']  for r in results) / n,
    'schema_keys_rate':        sum(r['keys']   for r in results) / n,
    'cc_overlap_mean':         sum(r['cc']     for r in results) / n,
    'evidence_grounding_rate': sum(r['ground'] for r in results) / n,
}
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 13 — qualitative inspection
for i in range(min(2, len(eval_rows))):
    print('='*72)
    print('TRANSCRIPT:')
    print(eval_rows[i]['transcript'])
    print('\nPREDICTED:')
    print(preds[i])
    print('\nGOLD:')
    print(json.dumps(eval_rows[i]['soap'], indent=2))

In [ ]:
# Cell 14 — save adapter
ADAPTER_DIR = '/kaggle/working/adapter_v0'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved ->', ADAPTER_DIR)
!ls -la $ADAPTER_DIR

## v2 roadmap

Once the spike confirms the pipeline:

1. **Realistic transcripts** — 2000-2500 token messy dialogues (synthetic via GPT-4o, then hand-corrected sample).
2. **Multi-template training** — add referral letter spec; train on mixture; eval on held-out spec to prove template-awareness.
3. **Frozen eval set** — 50 examples (20 hand-curated + 30 synthetic-corrected), built before training.
4. **Stronger metrics** — per-field F1, schema validation via jsonschema, judge-model semantic scoring.
5. **Guardrails** — Outlines/jsonschema-constrained decoding, PII de-identification pre-step.
6. **DPO/RSFT** — once SFT plateaus.